In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def projection(v : np.array):
    """ projette un vecteur quelconque sur 
    le simplexe pour la norme euclidienne"""

    upper_bound : np.float = np.max(v)
    lower_bound : np.float = np.max(v) - 1 
    theta : np.float = 0.5*(lower_bound + upper_bound) 
    epsilon : np.float = 0.00001
    regression : np.array = np.maximum(0, v - theta)
    error : np.float = (np.sum(regression) - 1)

    while (error)**2 > epsilon :
        if error >= 0:
            lower_bound = theta
            theta = 0.5*(lower_bound + upper_bound)

        else :
            upper_bound = theta
            theta = 0.5*(lower_bound + upper_bound)

        regression = np.maximum(0, v - theta)
        error = (np.sum(regression) - 1)
    return regression

def duality_gap(x, y, A):
    """
    Calcule le Duality Gap pour un couple (x, y).
    Gap(x, y) = max_y' (x^T A y') - min_x' (x'^T A y)
    """
    # x^T A donne un vecteur ligne. Le max de ce vecteur est la meilleure réponse de y.
    # A y donne un vecteur colonne. Le min de ce vecteur est la meilleure réponse de x.
    val_max_y = np.max(x @ A) 
    val_min_x = np.min(A @ y)
    return val_max_y - val_min_x

In [1]:
class GameOptimizer:
    def __init__(self, x_init, y_init, A, eta):
        self.x = x_init.copy()
        self.y = y_init.copy()
        # Pour l'optimisme, on a besoin des gradients à t-1
        # Au début (t=0), on peut supposer que g_{t-1} = g_t ou 0.
        # Ici on initialise les "prev" comme les actuels pour simplifier le pas 1.
        self.x_prev = x_init.copy()
        self.y_prev = y_init.copy()
        self.grad_x_prev = np.zeros_like(x_init)
        self.grad_y_prev = np.zeros_like(y_init)
        
        self.A = A
        self.eta = eta
        
        # Historique pour les courbes
        self.history_x = [x_init.copy()]
        self.history_y = [y_init.copy()]
        self.gaps = [duality_gap(x_init, y_init, A)]

    def _compute_gradients(self, x, y):
        # Gradient pour x (Minimizer) : A * y
        grad_x = self.A @ y
        # Gradient pour y (Maximizer) : A.T * x (on fait une montée de gradient)
        grad_y = self.A.T @ x 
        return grad_x, grad_y

    def step(self):
        raise NotImplementedError

class OGDA(GameOptimizer):
    """Optimistic Gradient Descent Ascent"""
    def step(self):
        # 1. Calcul des gradients actuels
        grad_x, grad_y = self._compute_gradients(self.x, self.y)
        
        # 2. Calcul des gradients "Optimistes"
        # M_t = 2 * g_t - g_{t-1}
        optim_grad_x = 2 * grad_x - self.grad_x_prev
        optim_grad_y = 2 * grad_y - self.grad_y_prev
        
        # 3. Mise à jour des variables avec PROJECTION
        # x veut minimiser -> on soustrait le gradient (Descente)
        self.x = projection_simplex(self.x - self.eta * optim_grad_x)
        # y veut maximiser -> on ajoute le gradient (Montée)
        self.y = projection_simplex(self.y + self.eta * optim_grad_y)
        
        # 4. Sauvegarde pour l'étape suivante
        self.grad_x_prev = grad_x
        self.grad_y_prev = grad_y
        
        # 5. Stockage
        self.history_x.append(self.x.copy())
        self.history_y.append(self.y.copy())
        self.gaps.append(duality_gap(self.x, self.y, self.A))

class OMWU(GameOptimizer):
    """Optimistic Multiplicative Weights Update"""
    def step(self):
        # 1. Calcul des gradients actuels
        grad_x, grad_y = self._compute_gradients(self.x, self.y)
        
        # 2. Gradients Optimistes
        optim_grad_x = 2 * grad_x - self.grad_x_prev
        optim_grad_y = 2 * grad_y - self.grad_y_prev
        
        # 3. Mise à jour Multiplicative (Exponentielle) [cite: 2177]
        # x minimisation : multiplie par exp(-eta * gradient)
        self.x = self.x * np.exp(-self.eta * optim_grad_x)
        self.x /= np.sum(self.x) # Renormalisation (Projection KL)
        
        # y maximisation : multiplie par exp(+eta * gradient)
        self.y = self.y * np.exp(self.eta * optim_grad_y)
        self.y /= np.sum(self.y)
        
        # 4. Sauvegarde
        self.grad_x_prev = grad_x
        self.grad_y_prev = grad_y
        
        # 5. Stockage
        self.history_x.append(self.x.copy())
        self.history_y.append(self.y.copy())
        self.gaps.append(duality_gap(self.x, self.y, self.A))